# AObench: how far can the oracle's 5-layer window move?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project/blob/main/notebooks/aobench_layer_window_sweep.ipynb)

This notebook loads Qwen3-8B, the Activation Oracle of
[Building Better Activation Oracles](https://arxiv.org/abs/2606.02609) and the paper's benchmark,
AObench. It then runs our first experiment: the oracle was trained to read layers 21 to 25, and we
measure how its AObench score changes when we feed it another window of 5 consecutive layers.

The code lives in `src/procom/oracle/aobench.py`; the Colab workflow is explained in
`tutorial_colab_oracle.ipynb`.

**Before running**

1. Runtime: *Change runtime type → A100 GPU*. Qwen3-8B is loaded in bf16 (16 GB), a T4 is too small.
2. No API key is needed. Five of the nine evals are graded by an LLM judge: the paper uses Claude
   Sonnet 4.6 through a paid API, we run an open-weights judge (Qwen3-14B in 4-bit, about 10 GB) on
   the same GPU instead.
3. Results are written to Google Drive, so a disconnected runtime loses nothing: reopen the
   notebook, *Run all*, and the sweep resumes where it stopped.
4. Leave `QUICK = True` for a first pass (10 examples per eval, a few minutes) to check that
   everything runs, then set it to `False` for the real experiment.

## 0. Setup

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project.git"
BRANCH = "main"  # put your branch here to test code that is not merged yet

if IN_COLAB:
    # Clone the repo (or update it), then install the procom package and its dependencies
    !git clone -q {REPO_URL} /content/procom 2>/dev/null
    !cd /content/procom && git fetch -q && git checkout -q {BRANCH} && git pull -q
    !pip install -q -e /content/procom
    sys.path.insert(0, "/content/procom/src")  # makes procom importable without restarting

In [ ]:
import os

import torch

from procom.oracle import aobench

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100 GPU"
print(torch.cuda.get_device_name(0))

QUICK = True  # True: 10 examples per eval, to test the pipeline. False: the full benchmark.
OUTPUT_DIR = "outputs/aobench_window_sweep"

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/procom/aobench_window_sweep"
    try:  # optional: a Hugging Face token only makes the downloads faster
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:  # secret missing, or access not granted
        pass
if QUICK:
    OUTPUT_DIR += "_quick"

# AObench is not a pip package: this clones it at the commit used by the paper's code
aobench.install_aobench("/content/aobench" if IN_COLAB else "outputs/aobench")
print("Results in:", OUTPUT_DIR)

## 1. Qwen3-8B and the oracle

As in the tutorial, the oracle is a LoRA adapter on Qwen3-8B itself, so one model plays both roles.
What changes with this paper's oracle (`ceselder/qwen3-8b-ao-v3-best`):

- it reads **5 layers at once**, 21 to 25, instead of one layer at 50 % depth. Its prompt contains
  one `Layer: N` line followed by the ` ?` placeholders for each layer;
- by default AObench gives it the activations of the last 20 tokens of the text.

The injection is unchanged: the vectors are added, norm-matched, onto the ` ?` tokens at the output
of layer 1. `ask_oracle` goes through AObench's own code, so it does exactly what the benchmark does
for one example. Its `layers` argument moves the window.

The second cell loads the judge. With 80 GB of GPU memory, `Qwen/Qwen3-32B` (about 20 GB in 4-bit)
is a stronger choice; a standard Colab A100 has 40 GB, where Qwen3-14B is the safe one.

In [ ]:
model, tokenizer = aobench.load_model_and_tokenizer()
print(f"{aobench.MODEL_NAME}: {model.config.num_hidden_layers} layers")
print("Oracle:", aobench.ORACLE_LORA)

In [ ]:
JUDGE_MODEL = "Qwen/Qwen3-14B"

aobench.use_local_judge(JUDGE_MODEL)
EVALS = aobench.default_evals()
print("Evals:", EVALS)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.0f} GB")

In [ ]:
target = [
    {
        "role": "user",
        "content": "The philosopher who drank hemlock taught a student who founded an academy. "
        "That student's most famous pupil was",
    }
]
question = "Can you name which person the model is thinking about?"

for start in (21, 13, 31):
    layers = aobench.window_layers(start)
    answer = aobench.ask_oracle(model, tokenizer, target, question, layers)
    print(f"layers {layers[0]}-{layers[-1]}:", answer)

## 2. AObench on the trained window

AObench is a set of evals, each asking the oracle questions about activations of Qwen3-8B whose
answer is known. We run the nine that make up the paper's aggregate score:

| Eval | What the oracle must do | Scoring |
|---|---|---|
| `number_prediction` | predict the number the model is about to output | exact match |
| `mmlu_prediction` | say whether the model will answer an MMLU question correctly | ROC AUC |
| `missing_info` | detect that the model lacks information | ROC AUC |
| `sycophancy` | detect that the model agrees because of the user | ROC AUC |
| `backtracking` | explain what the model is uncertain about | judge, 1 to 5 |
| `vagueness` | give specific rather than generic answers | judge |
| `domain_confusion` | name the right problem domain | judge |
| `activation_sensitivity` | answer differently when the same tokens hide different states | judge |
| `hallucination` | avoid confident, obviously wrong claims | judge |

`taboo` and `personaqa` are left out: they need extra fine-tuned target models.

The **aggregate score** is the mean over the evals of a normalized score between 0 and 1
(for the ROC AUC evals, 0 is chance; the 1 to 5 grades are rescaled).

We differ from the paper's setup on three points, so our scores are not directly comparable with
its numbers, but they are comparable from one window to the next:

- the judge is our local model, not Claude Sonnet 4.6;
- the oracle's prompt names the layers it is really given (`prompt_layers="actual"`);
- Qwen3-8B's activations are read with the oracle's LoRA disabled (`target_activations="base"`),
  as they were when the oracle was trained. AObench as published leaves that LoRA active while
  reading them.

`run_sweep` runs AObench for a list of windows. Here the list holds only the trained window, which
gives our reference score and the time one run of the benchmark takes on this GPU.

In [ ]:
sweep_settings = dict(
    output_dir=OUTPUT_DIR,
    evals=EVALS,
    sample_limits=aobench.QUICK_SAMPLE_LIMITS if QUICK else None,
    # The oracle's prompt names the layers it is really given ("trained" would always say 21-25)
    prompt_layers="actual",
    # Activations read on Qwen3-8B with every LoRA disabled ("aobench": as the benchmark does)
    target_activations="base",
)
TRAINED_START = aobench.TRAINED_WINDOW[0]

aobench.run_sweep(model, tokenizer, starts=[TRAINED_START], **sweep_settings)
baseline = aobench.collect_results(OUTPUT_DIR, starts=[TRAINED_START])
baseline

## 3. Moving the window

We keep the window 5 layers wide and slide it one layer at a time, from layers 13-17 to layers
31-35 (the last layer of the model): 19 windows, 8 on the shallow side of the trained window and 10
on the deep side. Consecutive windows share 4 layers. The windows are run from the trained one
outwards (20, 22, 19, 23, ...), so an interrupted run still gives a complete curve around the
reference.

To shorten the run, raise `STRIDE` (2 gives 10 windows) or narrow `FIRST_START` / `LAST_START`.
The cell below turns the timing of section 2 into an estimate for the whole sweep.

In [ ]:
FIRST_START, LAST_START, STRIDE = 13, 31, 1

starts = aobench.window_schedule(FIRST_START, LAST_START, STRIDE)
minutes_per_window = baseline["minutes"].sum()
print(f"{len(starts)} windows, in this order:", [f"{s}-{s + 4}" for s in starts])
print(
    f"One window took {minutes_per_window:.0f} min: about "
    f"{minutes_per_window * (len(starts) - 1) / 60:.1f} h for the {len(starts) - 1} other windows."
)

In [ ]:
aobench.run_sweep(model, tokenizer, starts=starts, **sweep_settings)

## 4. Results

`aggregate` is the AObench aggregate score of each window; the other columns are the normalized
scores of the evals. The second table gives the loss relative to the trained window.

In [ ]:
results = aobench.collect_results(OUTPUT_DIR, starts=starts)
results.to_csv(f"{OUTPUT_DIR}/results.csv", index=False)

table = aobench.aggregate_scores(results)
table.round(3)

In [ ]:
reference = table.loc[TRAINED_START, "aggregate"]
loss = (table[["aggregate"]] - reference).rename(columns={"aggregate": "change"})
loss["relative"] = (loss["change"] / reference).map("{:+.0%}".format)
loss.round(3)

In [ ]:
%matplotlib inline
fig = aobench.plot_sweep(results)
fig.savefig(f"{OUTPUT_DIR}/sweep.png", dpi=150)